# Stock Market Sentiment Analysis Using FastAPI

This notebook demonstrates a sentiment analysis microservice for financial news.
We will build the app step by step in separate cells for classroom demonstration.

In [2]:
import sys
!{sys.executable} -m pip install fastapi uvicorn vaderSentiment textblob pydantic



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: C:\Users\sujal kumar\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [5]:
# Cell 2: Import required libraries
import re
from typing import List, Optional

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob


ModuleNotFoundError: No module named 'vaderSentiment'

In [ ]:
# Cell 3: Create the FastAPI app
app = FastAPI(
    title="Stock Market Financial Sentiment Analysis Engine",
    description="NLP microservice for analyzing Indian stock market financial news and sentiment",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("FastAPI app created successfully")

In [ ]:
# Cell 4: Create a custom financial dictionary
vader_analyzer = SentimentIntensityAnalyzer()

FINANCIAL_LEXICON = {
    "surge": 1.5, "surged": 1.5, "jump": 1.2, "jumped": 1.2, "rally": 1.6, "all-time high": 2.0,
    "profit": 1.5, "revenue growth": 1.8, "bullish": 1.5, "outperform": 1.6, "upgrade": 1.5,
    "record high": 2.0, "dividend": 1.2, "acquisition": 1.0, "expansion": 1.2, "beat estimates": 1.8,
    "strong buy": 2.0, "plummet": -1.8, "plummeted": -1.8, "crash": -2.0, "slump": -1.5,
    "loss": -1.5, "decline": -1.2, "bearish": -1.5, "downgrade": -1.6, "missed estimates": -1.7,
    "investigation": -1.5, "fraud": -2.5, "default": -2.5, "debt crisis": -2.0, "penalty": -1.4,
    "layoffs": -1.3, "resignation": -1.0, "sell-off": -1.6
}

vader_analyzer.lexicon.update(FINANCIAL_LEXICON)
print("Custom lexicon loaded:", len(FINANCIAL_LEXICON), "terms")

In [ ]:
# Cell 5: Define request and response models
class SentimentRequest(BaseModel):
    text: str
    symbol: Optional[str] = None

class BatchSentimentRequest(BaseModel):
    articles: List[SentimentRequest]

class SentimentResponse(BaseModel):
    text: str
    symbol: Optional[str] = None
    score: float
    confidence: float
    label: str
    positivePct: float
    neutralPct: float
    negativePct: float
    keyFactors: List[str]

class AggregatedResponse(BaseModel):
    symbol: str
    overallSentiment: str
    sentimentScore: float
    confidence: float
    positivePercentage: float
    neutralPercentage: float
    negativePercentage: float
    sentimentTrend: str
    totalArticles: int

print("Models defined successfully")

In [ ]:
# Cell 6: Health check endpoint
@app.get("/health")
def health_check():
    return {"status": "healthy", "service": "Kotak Stock Market NLP Sentiment Service"}

health_check()

In [ ]:
# Cell 7: Build the main sentiment analysis function
@app.post("/analyze", response_model=SentimentResponse)
def analyze_text(request: SentimentRequest):
    text = request.text
    if not text or len(text.strip()) == 0:
        raise HTTPException(status_code=400, detail="Text cannot be empty")

    # Step 1: VADER sentiment score
    vader_scores = vader_analyzer.polarity_scores(text)
    compound = vader_scores["compound"]

    # Step 2: TextBlob sentiment score for additional signal
    blob = TextBlob(text)
    tb_polarity = blob.sentiment.polarity
    tb_subjectivity = blob.sentiment.subjectivity

    # Step 3: Combine both scores
    combined_score = round((compound * 0.7) + (tb_polarity * 0.3), 4)
    combined_score = max(-1.0, min(1.0, combined_score))

    # Step 4: Convert into positive/neutral/negative percentages
    pos = max(0.0, vader_scores["pos"])
    neu = max(0.0, vader_scores["neu"])
    neg = max(0.0, vader_scores["neg"])
    total = pos + neu + neg

    if total > 0:
        pos_pct = round((pos / total) * 100, 2)
        neu_pct = round((neu / total) * 100, 2)
        neg_pct = round((neg / total) * 100, 2)
    else:
        pos_pct, neu_pct, neg_pct = 33.33, 33.34, 33.33

    # Step 5: Label based on final score
    if combined_score >= 0.15:
        label = "POSITIVE"
    elif combined_score <= -0.15:
        label = "NEGATIVE"
    else:
        label = "NEUTRAL"

    # Step 6: Confidence score
    confidence = round(min(1.0, abs(combined_score) + (1.0 - tb_subjectivity) * 0.3), 4)

    # Step 7: Detect important financial keywords
    words = re.findall(r"\b\w+\b", text.lower())
    key_factors = []
    for w in words:
        if w in FINANCIAL_LEXICON:
            impact = "Positive" if FINANCIAL_LEXICON[w] > 0 else "Negative"
            key_factors.append(f"Keyword '{w.capitalize()}' detected ({impact} catalyst)")

    if not key_factors:
        key_factors.append("General market news commentary")

    return SentimentResponse(
        text=text,
        symbol=request.symbol,
        score=combined_score,
        confidence=confidence,
        label=label,
        positivePct=pos_pct,
        neutralPct=neu_pct,
        negativePct=neg_pct,
        keyFactors=list(set(key_factors))[:5]
    )

sample = SentimentRequest(
    text="Reliance Industries surged strongly after record quarterly profit and revenue growth.",
    symbol="RELIANCE"
)

analyze_text(sample)

In [ ]:
# Cell 8: Add batch sentiment endpoint
@app.post("/analyze-batch", response_model=List[SentimentResponse])
def analyze_batch(request: BatchSentimentRequest):
    results = []
    for item in request.articles:
        results.append(analyze_text(item))
    return results

batch_request = BatchSentimentRequest(
    articles=[
        SentimentRequest(text="Infosys reported strong earnings and raised guidance.", symbol="INFY"),
        SentimentRequest(text="Banking stocks fell sharply after debt concerns and weak margins.", symbol="BANK"),
    ]
)

analyze_batch(batch_request)

In [ ]:
# Cell 9: Add aggregated sentiment endpoint
@app.post("/aggregate", response_model=AggregatedResponse)
def aggregate_symbol_sentiment(symbol: str, articles: List[SentimentRequest]):
    if not articles:
        return AggregatedResponse(
            symbol=symbol,
            overallSentiment="NEUTRAL",
            sentimentScore=0.0,
            confidence=0.5,
            positivePercentage=33.33,
            neutralPercentage=33.34,
            negativePercentage=33.33,
            sentimentTrend="STABLE",
            totalArticles=0
        )

    evaluated = [analyze_text(a) for a in articles]
    scores = [e.score for e in evaluated]
    confidences = [e.confidence for e in evaluated]

    avg_score = round(sum(scores) / len(scores), 4)
    avg_confidence = round(sum(confidences) / len(confidences), 4)

    pos_count = sum(1 for e in evaluated if e.label == "POSITIVE")
    neu_count = sum(1 for e in evaluated if e.label == "NEUTRAL")
    neg_count = sum(1 for e in evaluated if e.label == "NEGATIVE")

    total = len(evaluated)
    pos_pct = round((pos_count / total) * 100, 2)
    neu_pct = round((neu_count / total) * 100, 2)
    neg_pct = round((neg_count / total) * 100, 2)

    if avg_score >= 0.15:
        overall = "POSITIVE"
    elif avg_score <= -0.15:
        overall = "NEGATIVE"
    else:
        overall = "NEUTRAL"

    trend = "BULLISH_SURGE" if pos_pct >= 60 else ("BEARISH_SLUMP" if neg_pct >= 60 else "STABLE")

    return AggregatedResponse(
        symbol=symbol,
        overallSentiment=overall,
        sentimentScore=avg_score,
        confidence=avg_confidence,
        positivePercentage=pos_pct,
        neutralPercentage=neu_pct,
        negativePercentage=neg_pct,
        sentimentTrend=trend,
        totalArticles=total
    )

aggregate_symbol_sentiment(
    symbol="TCS",
    articles=[
        SentimentRequest(text="TCS reported strong profit growth and a positive outlook.", symbol="TCS"),
        SentimentRequest(text="TCS faced minor concerns due to slow order growth.", symbol="TCS"),
    ]
)

In [ ]:
# Cell 10: Run the FastAPI app
if __name__ == "__main__":
    import uvicorn
    uvicorn.run("main:app", host="0.0.0.0", port=8000, reload=True)

print("Notebook ready. Use uvicorn or FastAPI TestClient to test endpoints.")